In [5]:
import networkx as nx
from itertools import product, combinations_with_replacement

def get_hypercube_edges(d):
    """Generates the full edge set of a d-dimensional {-1, 1}^d hypercube."""
    # Vertices are tuples of length d with values -1 and 1
    vertices = list(product([-1, 1], repeat=d))
    edges = []
    for i in range(len(vertices)):
        for j in range(i + 1, len(vertices)):
            # Edges exist between vertices differing by exactly one coordinate
            diff = sum(1 for x, y in zip(vertices[i], vertices[j]) if x != y)
            if diff == 1:
                edges.append(tuple([tuple(vertices[i]), tuple(vertices[j])]))
    return edges

import itertools


def get_vertex_set_from_edges(edge_set):
    s=set()
    for edge in edge_set:
        s.add(tuple(edge[0]))
        s.add(tuple(edge[1]))
    return s

def apply_permutation(vertex_set, p):
    d=len(p)
    return [[v[p[i]] for i in range(d)] for v in vertex_set]

def get_transform_v1_v2(v1, v2):
    diff_list = [v1[i] * v2[i] for i in range(len(v1))]
    return lambda v: [v[i] * diff_list[i] for i in range(len(v))]


def get_if_original_set(new_lists, old_set):
    for l in new_lists:
        if tuple(l) not in old_set:
            return False
    return True

def count_symmetries(vertex_set):
    sym = 0
    d = len(list(vertex_set)[0])
    vertex_set = list(set([tuple(v) for v in vertex_set]))
    for p in itertools.permutations(range(d)):
        list_00 = apply_permutation(vertex_set, p)
        v1 = list_00[0]
        for v in vertex_set:
            sign_map = get_transform_v1_v2(v1, v)
            list_01 = list(map(sign_map, list_00))
            sym+=get_if_original_set(list_01, vertex_set)
    return sym

#print(count_symmetries([[1,1,1]]))

print(count_symmetries([[1,1,1],[-1,1,1], [1,-1,1], [-1,-1,1]]))





8


In [32]:
#don't feel like transforming into torch and back rn

def for_loop_hyperplane_intersection(h, edge_subset):
    intersected_edges = []
    for v1, v2 in edge_subset:
        val1 = sum([h[i]*v1_val for i,v1_val in enumerate(v1)])-h[-1]
        val2 = sum([h[i]*v2_val for i,v2_val in enumerate(v2)])-h[-1]
        if val1*val2<0:
            intersected_edges.append(tuple([v1,v2]))
    return intersected_edges 


        

In [3]:
from collections import Counter 

def deg_seq(edge_subset):
    v_degs = Counter()
    for v1,v2 in edge_subset:
        v_degs[v1]+=1
        v_degs[v2]+=1
    degs = Counter()
    for _, deg in v_degs.items():
        degs[deg]+=1
    return tuple([tuple([deg, deg_count]) for deg, deg_count in degs.items()])
    

In [20]:
import pandas as pd
import random, time

d=4

filename = f"symmetries_tracked_{d}.csv"

h_limit=10
b_limit=10

chance = .01

edge_set = set(get_hypercube_edges(d))
edges = len(edge_set)
rows=[]

i=0
next_i=1
for h_coef in combinations_with_replacement(range(h_limit),d):
    for b in range(b_limit):
        # i+=1
        # if i!=next_i:
        #     continue
        # else: 
        #     next_i += random.randint(10,100)
        
        plane = h_coef+tuple([b])
        t_s = time.time()
        print(f"doing {plane}")
        intersected_edges = set(for_loop_hyperplane_intersection(plane, edge_set))
        non_intersected_edges = edge_set.difference(intersected_edges)

        vertices_in = get_vertex_set_from_edges(intersected_edges)
        vertices_out = get_vertex_set_from_edges(non_intersected_edges)
        if len(vertices_in)==0:
            continue
        t_int = time.time()
        print(f"{t_int-t_s} to get intersected edges")
        row_data = {"hyperplane": tuple(plane),
                   "edges_cut": len(intersected_edges),
                   "edges_not_cut": edges-len(intersected_edges),
                   "symmetries_inside": count_symmetries(vertices_in),
                   "symmetries_outside": count_symmetries(vertices_out),
                   "struct_inside": deg_seq(intersected_edges), 
                   "struct_outside": deg_seq(non_intersected_edges)}
        print(f"{time.time()-t_int} to get symmetries")
        print(row_data)
        rows.append(row_data)

sym_df = pd.DataFrame(rows)
#sym_df.to_csv(filename)
sym_df.head()


doing (0, 0, 0, 0, 0)
doing (0, 0, 0, 0, 1)
doing (0, 0, 0, 0, 2)
doing (0, 0, 0, 0, 3)
doing (0, 0, 0, 0, 4)
doing (0, 0, 0, 0, 5)
doing (0, 0, 0, 0, 6)
doing (0, 0, 0, 0, 7)
doing (0, 0, 0, 0, 8)
doing (0, 0, 0, 0, 9)
doing (0, 0, 0, 1, 0)
0.00011777877807617188 to get intersected edges
0.014216899871826172 to get symmetries
{'hyperplane': (0, 0, 0, 1, 0), 'edges_cut': 8, 'edges_not_cut': 24, 'symmetries_inside': 384, 'symmetries_outside': 384, 'struct_inside': ((1, 16),), 'struct_outside': ((3, 16),)}
doing (0, 0, 0, 1, 1)
doing (0, 0, 0, 1, 2)
doing (0, 0, 0, 1, 3)
doing (0, 0, 0, 1, 4)
doing (0, 0, 0, 1, 5)
doing (0, 0, 0, 1, 6)
doing (0, 0, 0, 1, 7)
doing (0, 0, 0, 1, 8)
doing (0, 0, 0, 1, 9)
doing (0, 0, 0, 2, 0)
4.649162292480469e-05 to get intersected edges
0.007804155349731445 to get symmetries
{'hyperplane': (0, 0, 0, 2, 0), 'edges_cut': 8, 'edges_not_cut': 24, 'symmetries_inside': 384, 'symmetries_outside': 384, 'struct_inside': ((1, 16),), 'struct_outside': ((3, 16),)}
doi

,hyperplane,edges_cut,edges_not_cut,symmetries_inside,symmetries_outside,struct_inside,struct_outside
0,"(0, 0, 0, 1, 0)",8,24,384,384,"((1, 16),)","((3, 16),)"
1,"(0, 0, 0, 2, 0)",8,24,384,384,"((1, 16),)","((3, 16),)"
2,"(0, 0, 0, 2, 1)",8,24,384,384,"((1, 16),)","((3, 16),)"
3,"(0, 0, 0, 3, 0)",8,24,384,384,"((1, 16),)","((3, 16),)"
4,"(0, 0, 0, 3, 1)",8,24,384,384,"((1, 16),)","((3, 16),)"


In [8]:
sym_df["symmetries_inside"].unique()

array([384,  16,  48,   4,  12,  24,   2,   8,   6])

In [21]:
sym_df["symmetries_outside"].unique()

array([384,  24])

In [9]:
sym_df["struct_inside"].unique()

array([((1, 16),), ((1, 8), (2, 4)), ((1, 8),), ((1, 4), (2, 2)),
       ((2, 6), (1, 8)), ((1, 8), (2, 2)), ((1, 4),), ((2, 12),),
       ((2, 6), (1, 4)), ((1, 14),), ((1, 11), (2, 3), (3, 1)),
       ((1, 6), (3, 2)), ((1, 6), (2, 2)), ((2, 7), (1, 2)),
       ((1, 8), (2, 3)), ((1, 6), (2, 3)), ((2, 4), (1, 8)),
       ((2, 5), (3, 1), (1, 7)), ((1, 6), (2, 2), (3, 2)),
       ((2, 6), (1, 6), (3, 2)), ((1, 5), (3, 3), (2, 3)),
       ((1, 5), (2, 4), (3, 1)), ((2, 8), (3, 2), (1, 2)),
       ((2, 6), (3, 1), (1, 1)), ((2, 5), (3, 1), (1, 5)),
       ((1, 4), (3, 2), (2, 1)), ((1, 4), (4, 1)),
       ((2, 4), (1, 5), (3, 1)), ((2, 9), (1, 2)), ((2, 6), (3, 4)),
       ((2, 7), (3, 2)), ((2, 3), (1, 3), (3, 1))], dtype=object)

In [18]:
sym_df[sym_df["symmetries_inside"]>sym_df["symmetries_outside"]]

,hyperplane,edges_cut,edges_not_cut,symmetries_inside,symmetries_outside,struct_inside,struct_outside


In [38]:
from itertools import chain, combinations

def powerset(iterable):
    """powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"""
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s) + 1))

planes = [[1,1,1,3,3,-4,0],
                        [-2,-2,-2,3,3,-1,0],
                        [3,3,3,1,1,-4,0],
                        [-1,-1,-1,3,3,6,0],
                        [3,3,3,1,1,8,0]]
edge_set = set(get_hypercube_edges(len(planes[0])-1))
for p in planes:
    
    intersected_edges = for_loop_hyperplane_intersection(p, edge_set)
    print(len(intersected_edges))

for p_list in powerset(planes):
    print(len(p_list), " planes")
    print(p_list)
    intersected_edges = set()
    for p in p_list:
        intersected_edges = intersected_edges.union(set(for_loop_hyperplane_intersection(p, edge_set)))
    
    non_intersected_edges = edge_set.difference(intersected_edges)

    vertices_in = get_vertex_set_from_edges(intersected_edges)
    vertices_out = get_vertex_set_from_edges(non_intersected_edges)
    if len(vertices_in)==0:
        continue
    t_int = time.time()
    print(f"{t_int-t_s} to get intersected edges")
    row_data = {"edges_cut": len(intersected_edges),
               "edges_not_cut": edges-len(intersected_edges),
               "symmetries_inside": count_symmetries(vertices_in),
               "symmetries_outside": count_symmetries(vertices_out),
               "struct_inside": deg_seq(intersected_edges), 
               "struct_outside": deg_seq(non_intersected_edges)}
    print(f"{time.time()-t_int} to get symmetries")
    print(row_data)
    rows.append(row_data)



52
60
60
52
48
0  planes
()
1  planes
([1, 1, 1, 3, 3, -4, 0],)
3766.463777780533 to get intersected edges
4.117760181427002 to get symmetries
{'edges_cut': 52, 'edges_not_cut': -20, 'symmetries_inside': 72, 'symmetries_outside': 46080, 'struct_inside': ((3, 6), (4, 2), (2, 36), (1, 6)), 'struct_outside': ((6, 14), (4, 36), (5, 6), (3, 6), (2, 2))}
1  planes
([-2, -2, -2, 3, 3, -1, 0],)
3770.58193731308 to get intersected edges
3.8326871395111084 to get symmetries
{'edges_cut': 60, 'edges_not_cut': -28, 'symmetries_inside': 24, 'symmetries_outside': 46080, 'struct_inside': ((3, 32), (2, 6), (4, 2), (1, 4)), 'struct_outside': ((6, 20), (4, 6), (3, 32), (5, 4), (2, 2))}
1  planes
([3, 3, 3, 1, 1, -4, 0],)
3774.415102005005 to get intersected edges
3.8973097801208496 to get symmetries
{'edges_cut': 60, 'edges_not_cut': -28, 'symmetries_inside': 96, 'symmetries_outside': 46080, 'struct_inside': ((3, 24), (4, 6), (2, 6), (1, 12)), 'struct_outside': ((3, 24), (6, 16), (4, 6), (5, 12), (2, 6)

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import random, time

d=5

filename = f"symmetries_tracked_{d}.csv"

h_limit=10
b_limit=10

chance = .01

edge_set = set(get_hypercube_edges(d))
edges = len(edge_set)
rows=[]

i=0
next_i=1
for h_coef in combinations_with_replacement(range(h_limit),d):
    for b in range(b_limit):
        i+=1
        if i!=next_i:
            continue
        else: 
            next_i += random.randint(10,100)
        
        plane = h_coef+tuple([b])
        t_s = time.time()
        print(f"doing {plane}")
        intersected_edges = set(for_loop_hyperplane_intersection(plane, edge_set))
        non_intersected_edges = edge_set.difference(intersected_edges)

        vertices_in = get_vertex_set_from_edges(intersected_edges)
        vertices_out = get_vertex_set_from_edges(non_intersected_edges)
        if len(vertices_in)==0:
            continue
        t_int = time.time()
        print(f"{t_int-t_s} to get intersected edges")
        row_data = {"hyperplane": tuple(plane),
                   "edges_cut": len(intersected_edges),
                   "edges_not_cut": edges-len(intersected_edges),
                   "symmetries_inside": count_symmetries(vertices_in),
                   "symmetries_outside": count_symmetries(vertices_out),
                   "struct_inside": deg_seq(intersected_edges), 
                   "struct_outside": deg_seq(non_intersected_edges)}
        print(f"{time.time()-t_int} to get symmetries")
        print(row_data)
        rows.append(row_data)

sym_df = pd.DataFrame(rows)
#sym_df.to_csv(filename)
sym_df.head()
